# Fine-Tuning Orchestrator

Notebook ini mengatur pelatihan model Bi-Encoder dan Cross-Encoder untuk CV Matching System.

## Struktur Direktori
- `data/training/`: Dataset CSV (Triplet & Pairs)
- `training/scripts/`: Script pelatihan Python
- `models/`: Output model terlatih

## Langkah Persiapan
1. Pastikan dataset CSV tersedia di `data/training/`
2. Install dependencies: `pip install -r backend/requirements.txt`

In [ ]:
# Setup Paths
import os
import sys

# Add training scripts to path
sys.path.append('training/scripts')

# Define paths
DATA_DIR = '/home/kuliah/intern/dicoding/caps-final/data/training'
MODEL_DIR = '/home/kuliah/intern/dicoding/caps-final/models'
SCRIPT_DIR = '/home/kuliah/intern/dicoding/caps-final/training/scripts'
SKILLS_DIR = '/home/kuliah/intern/dicoding/caps-final/backend/app/core/skills'
DATASET_SCRIPT = '/home/kuliah/intern/dicoding/caps-final/training/scripts/generate_dataset.py'
DATASET_TEMPLATE = '/home/kuliah/intern/dicoding/caps-final/training/templates'

# Check directory structure
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(MODEL_DIR, exist_ok=True)

print(f"Data directory: {os.path.abspath(DATA_DIR)}")
print(f"Model directory: {os.path.abspath(MODEL_DIR)}")
print(f"Script directory: {os.path.abspath(SCRIPT_DIR)}")

## Phase 1: Data Preparation

Dataset yang diperlukan:
1. `bi_encoder_train.csv` (Triplet: anchor, positive, negative)
2. `cross_encoder_train.csv` (Pairs: cv_text, jd_text, label)

Anda dapat menggunakan data sintetis untuk testing atau siapkan data real dari CV dan JD Anda.

In [ ]:
# Generate synthetic data using script
import subprocess

def generate_synthetic_data(num_triplets=2000, num_pairs=2000):
    print("Generating dataset using training/scripts/generate_dataset.py...")
    cmd = [
        'python', DATASET_SCRIPT,
        '--skills_dir', SKILLS_DIR,
        '--templates_dir', DATASET_TEMPLATE,
        '--output_dir', DATA_DIR,
        '--num_triplets', str(num_triplets),
        '--num_pairs', str(num_pairs)
    ]
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
    except Exception as e:
        print(f"Exception during generation: {e}")

# Run synthetic data generation
generate_synthetic_data()

# Check existing data
if os.path.exists(f'{DATA_DIR}/bi_encoder_train.csv'):
    print("Bi-Encoder training data found.")
else:
    print("Bi-Encoder training data NOT found.")

if os.path.exists(f'{DATA_DIR}/cross_encoder_train.csv'):
    print("Cross-Encoder training data found.")
else:
    print("Cross-Encoder training data NOT found.")

## Phase 2: Bi-Encoder Training

Melatih model Bi-Encoder menggunakan data triplet untuk menghasilkan embedding yang akurat.

In [ ]:
import subprocess

def train_bi_encoder():
    train_csv = f'{DATA_DIR}/bi_encoder_train.csv'
    output_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    
    if not os.path.exists(train_csv):
        print(f"Error: Training data not found at {train_csv}")
        return
    
    print(f"Training Bi-Encoder...")
    print(f"Input: {train_csv}")
    print(f"Output: {output_path}")
    
    # Run training script
    cmd = [
        'python', f'{SCRIPT_DIR}/train_bi_encoder.py',
        '--train_csv', train_csv,
        '--output_path', output_path,
        '--epochs', '5',
        '--batch_size', '16'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
        else:
            print("Bi-Encoder training completed successfully!")
    except Exception as e:
        print(f"Exception during training: {e}")

# Uncomment to run training
train_bi_encoder()


## Phase 3: Cross-Encoder Training

Melatih model Cross-Encoder untuk re-ranking kandidat dengan akurasi tinggi.

In [ ]:
def train_cross_encoder():
    train_csv = f'{DATA_DIR}/cross_encoder_train.csv'
    output_path = f'{MODEL_DIR}/cross-encoder-cv-matcher'
    
    if not os.path.exists(train_csv):
        print(f"Error: Training data not found at {train_csv}")
        return
    
    print(f"Training Cross-Encoder...")
    print(f"Input: {train_csv}")
    print(f"Output: {output_path}")
    
    # Run training script
    cmd = [
        'python', f'{SCRIPT_DIR}/train_cross_encoder.py',
        '--train_csv', train_csv,
        '--output_path', output_path,
        '--epochs', '3',
        '--batch_size', '16'
    ]
    
    try:
        result = subprocess.run(cmd, capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(f"Error: {result.stderr}")
        else:
            print("Cross-Encoder training completed successfully!")
    except Exception as e:
        print(f"Exception during training: {e}")

# Uncomment to run training
# train_cross_encoder()


## Phase 4: Model Verification

Verifikasi model yang telah dilatih tersimpan dengan benar.

In [ ]:
from sentence_transformers import SentenceTransformer, CrossEncoder

def verify_models():
    # Check Bi-Encoder
    bi_encoder_path = f'{MODEL_DIR}/bi-encoder-cv-matcher'
    if os.path.exists(bi_encoder_path):
        try:
            model = SentenceTransformer(bi_encoder_path)
            print(f"✓ Bi-Encoder loaded successfully from {bi_encoder_path}")
            # Test inference
            test_embedding = model.encode("Test sentence")
            print(f"  Embedding shape: {test_embedding.shape}")
        except Exception as e:
            print(f"✗ Error loading Bi-Encoder: {e}")
    else:
        print(f"✗ Bi-Encoder model not found at {bi_encoder_path}")
    
    # Check Cross-Encoder
    cross_encoder_path = f'{MODEL_DIR}/cross-encoder-cv-matcher'
    if os.path.exists(cross_encoder_path):
        try:
            model = CrossEncoder(cross_encoder_path)
            print(f"✓ Cross-Encoder loaded successfully from {cross_encoder_path}")
            # Test inference
            test_score = model.predict([("Test CV", "Test JD")])
            print(f"  Test score: {test_score}")
        except Exception as e:
            print(f"✗ Error loading Cross-Encoder: {e}")
    else:
        print(f"✗ Cross-Encoder model not found at {cross_encoder_path}")

verify_models()
